# Transformation

In diesem Notebook wird der bereinigte Datensatz in eine analysefreundliche Struktur überführt. Dabei werden Informationen, die sich aus den vorhandenen Daten
ableiten lassen, so aufbereitet, dass sie bei späteren Analysen nicht wiederholt berechnet werden müssen.

Dazu werden unter anderem Kalenderinformationen in einer separaten Datumstabelle bereitgestellt und der Umsatz je Rechnungsposition berechnet.

In [ ]:
from paths import CLEANED_DATA_DIR, TRANSFORMED_DATA_DIR
import pandas as pd

dataset = pd.read_parquet(CLEANED_DATA_DIR / "online_retail_II.parquet")

## Datumstabelle

Zeitbezogene Analysen benötigen häufig Merkmale wie Jahr, Quartal, Monat, Kalenderwoche oder Wochentag. Diese Informationen lassen sich zwar jederzeit
aus **InvoiceDate** berechnen, müssten dann jedoch bei jeder Analyse erneut abgeleitet werden.

Daher wird eine separate Datumstabelle erstellt, die jedes im Datensatz vorkommende Datum genau einmal enthält und die zugehörigen Kalendermerkmale
zentral bereitstellt.

Für Wochenanalysen werden **ISOYear** und **ISOWeek** verwendet. Das ISO-Jahr kann an Jahresgrenzen vom Kalenderjahr abweichen und ermöglicht zusammen
mit der ISO-Kalenderwoche eine eindeutige Zuordnung. Die Wochentage werden entsprechend der ISO-Konvention von Montag (`1`) bis Sonntag (`7`) nummeriert.

In [ ]:
invoice_date = dataset["InvoiceDate"].dt
iso_calendar = invoice_date.isocalendar()

date_dataset = pd.DataFrame({
    "Date": invoice_date.date,
    "Year": invoice_date.year,
    "Quarter": invoice_date.quarter,
    "Month": invoice_date.month,
    "MonthName": invoice_date.month_name(),
    "ISOYear": iso_calendar.year,
    "ISOWeek": iso_calendar.week,
    "Weekday": invoice_date.weekday + 1,
    "WeekdayName": invoice_date.day_name()
})

Jedes Datum kommt in der Datumstabelle nur einmal vor. Die Tabelle wird chronologisch sortiert und anschließend im Parquet-Format
gespeichert.

In [ ]:
date_dataset = (
    date_dataset
    .drop_duplicates(subset="Date")
    .sort_values("Date")
    .reset_index(drop=True)
)

date_dataset.to_parquet(TRANSFORMED_DATA_DIR / "date.parquet")

## Umsatz je Rechnungsposition

**Price** beschreibt den Stückpreis eines Artikels und reicht daher allein nicht aus, um den Umsatz einer Rechnungsposition zu bestimmen. Dieser ergibt sich aus
der verkauften Menge und dem jeweiligen Stückpreis.

Da der Umsatz eine zentrale Größe für spätere Auswertungen nach beispielsweise Datum, Produkt, Rechnung oder Land ist, wird er bereits während der Transformation
als **Revenue** bereitgestellt.

In [ ]:
dataset["Revenue"] = dataset["Quantity"] * dataset["Price"]

## Aufteilung des Rechnungszeitpunkts

**InvoiceDate** kombiniert zwei Informationen: das Kalenderdatum und die genaue Uhrzeit einer Buchung.

Das Datum wird für die Verknüpfung mit der Datumstabelle benötigt. Die Uhrzeit bleibt separat erhalten, damit auch Analysen innerhalb eines Tages möglich sind
und die zeitliche Reihenfolge von Rechnungen weiterhin nachvollzogen werden kann.

Daher wird **InvoiceDate** in **Date** und **Time** aufgeteilt.

In [ ]:
dataset["Time"] = invoice_date.time
dataset["Date"] = invoice_date.date

## Verknüpfung mit der Datumstabelle

Jede Rechnungsposition soll den zugehörigen Kalendertag eindeutig referenzieren können, ohne die Kalendermerkmale direkt in der Faktentabelle zu wiederholen.
Dafür erhält jeder Eintrag der Datumstabelle einen eindeutigen **date_key**.

Über das zuvor aus **InvoiceDate** extrahierte Datum wird dieser Schlüssel den Rechnungspositionen zugeordnet. Die eigentlichen Kalenderinformationen wie Jahr,
Monat, ISO-Kalenderwoche und Wochentag verbleiben damit ausschließlich in der Datumstabelle.

Nach erfolgreicher Zuordnung wird **Date** in der Faktentabelle nicht mehr benötigt. Die Datumsinformation wird dort durch **date_key** repräsentiert,
während **Time** die davon unabhängige Uhrzeit der Rechnung enthält.

In [ ]:
merged_dataset = dataset.merge(
    date_dataset.reset_index(names="date_key"),
    on="Date",
    how="inner"
)

dataset = merged_dataset[
    [
        "Invoice",
        "StockCode",
        "Description",
        "Quantity",
        "Price",
        "Customer ID",
        "Country",
        "Time",
        "Revenue",
        "date_key",
    ]
]